In [3]:
import pandas as pd
import ast
import math

# =========================
# 1. 파일 / 시트 설정
# =========================
file_path = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
sheet_name = "job_posting_processed"
output_path = "./data/공고원본_POSTING_ID포함_정리결과_협업툴_hold이동.xlsx"

# =========================
# 2. 협업 도구 / 업무 툴 키워드 사전
#    필요하면 여기서 추가/삭제
# =========================
HOLD_KEYWORDS = {
    "Git",
    "Figma",
    "Microsoft Excel",
    "Microsoft PowerPoint",
    "GitHub",
    "Microsoft Word",
    "Photoshop",
    "Microsoft Office",
    "UI/UX",
    "한컴",
    "Google Sheets",
    "Notion"
}

# 소문자 비교용
HOLD_KEYWORDS_LOWER = {x.lower() for x in HOLD_KEYWORDS}

# =========================
# 3. 리스트 파싱 함수
# =========================
def parse_list_cell(x):
    """
    문자열 형태의 리스트를 실제 list로 변환
    예: "['Java', 'Spring']" -> ['Java', 'Spring']
    """
    if pd.isna(x):
        return []
    
    if isinstance(x, list):
        return x
    
    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [parsed]
        except:
            # 혹시 리스트 문자열이 아니고 그냥 콤마 구분 문자열이면 대비
            return [item.strip() for item in x.split(",") if item.strip()]
    
    return []

# =========================
# 4. 문자열 생성 함수
# =========================
def list_to_str(lst):
    if not lst:
        return None
    return ", ".join(lst)

# =========================
# 5. hold 여부 판별 함수
# =========================
def is_hold_keyword(keyword):
    """
    exact match 기준.
    필요하면 contains 방식으로 확장 가능.
    """
    if keyword is None:
        return False
    
    k = str(keyword).strip()
    if k == "":
        return False
    
    return k.lower() in HOLD_KEYWORDS_LOWER

# =========================
# 6. 엑셀 읽기
# =========================
xls = pd.ExcelFile(file_path)
df = pd.read_excel(file_path, sheet_name=sheet_name)

# =========================
# 7. 컬럼 전처리
# =========================
df["TECH_STACK_NORMALIZED_LIST"] = df["TECH_STACK_NORMALIZED_LIST"].apply(parse_list_cell)
df["TECH_STACK_HOLD_LIST"] = df["TECH_STACK_HOLD_LIST"].apply(parse_list_cell)

# =========================
# 8. NORMALIZED -> HOLD 이동
# =========================
new_normalized_lists = []
new_hold_lists = []

for _, row in df.iterrows():
    normalized_list = row["TECH_STACK_NORMALIZED_LIST"]
    hold_list = row["TECH_STACK_HOLD_LIST"]

    moved_to_hold = [kw for kw in normalized_list if is_hold_keyword(kw)]
    remained_normalized = [kw for kw in normalized_list if not is_hold_keyword(kw)]

    # 기존 HOLD_LIST에 이어붙이기
    updated_hold = hold_list + moved_to_hold

    # 중복 제거(순서 유지)
    updated_hold = list(dict.fromkeys(updated_hold))
    remained_normalized = list(dict.fromkeys(remained_normalized))

    new_normalized_lists.append(remained_normalized)
    new_hold_lists.append(updated_hold)

# =========================
# 9. 결과 반영
# =========================
df["TECH_STACK_NORMALIZED_LIST"] = new_normalized_lists
df["TECH_STACK_HOLD_LIST"] = new_hold_lists

df["TECH_STACK_NORMALIZED"] = df["TECH_STACK_NORMALIZED_LIST"].apply(list_to_str)
df["TECH_STACK_HOLD"] = df["TECH_STACK_HOLD_LIST"].apply(list_to_str)

df["HAS_HOLD"] = df["TECH_STACK_HOLD_LIST"].apply(lambda x: len(x) > 0)

# =========================
# 10. 리스트 컬럼을 다시 문자열 형태로 저장
#     원본 형식 유지용
# =========================
df["TECH_STACK_NORMALIZED_LIST"] = df["TECH_STACK_NORMALIZED_LIST"].apply(str)
df["TECH_STACK_HOLD_LIST"] = df["TECH_STACK_HOLD_LIST"].apply(str)

# =========================
# 11. 저장
#     다른 시트는 그대로 두고 job_posting_processed만 교체
# =========================
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for s in xls.sheet_names:
        if s == sheet_name:
            df.to_excel(writer, sheet_name=s, index=False)
        else:
            temp_df = pd.read_excel(file_path, sheet_name=s)
            temp_df.to_excel(writer, sheet_name=s, index=False)

print("완료")
print(f"저장 파일: {output_path}")
print(f"HOLD 처리된 행 수: {df['HAS_HOLD'].sum()}")

완료
저장 파일: ./data/공고원본_POSTING_ID포함_정리결과_협업툴_hold이동.xlsx
HOLD 처리된 행 수: 226


In [7]:
import pandas as pd
import ast

# =========================
# 1. 파일 설정
# =========================
file_path = "./data/공고원본_POSTING_ID포함_정리결과.xlsx"
output_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영.xlsx"

sheet_job = "job_posting_processed"
sheet_norm_freq = "normalized_freq"
sheet_hold_freq = "hold_freq"

# =========================
# 2. HOLD로 보낼 키워드
# =========================
HOLD_KEYWORDS = {
    "Git",
    "Figma",
    "Microsoft Excel",
    "Microsoft PowerPoint",
    "GitHub",
    "Microsoft Word",
    "Photoshop",
    "Microsoft Office",
    "UI/UX",
    "한컴",
    "Google Sheets",
    "Notion"
}

HOLD_KEYWORDS_LOWER = {x.lower().strip() for x in HOLD_KEYWORDS}

# =========================
# 3. 유틸 함수
# =========================
def parse_list_cell(x):
    """
    문자열 형태의 리스트를 실제 list로 변환
    예: "['Java', 'Spring']" -> ['Java', 'Spring']
    """
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()
        if x == "":
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [parsed]
        except:
            return [item.strip() for item in x.split(",") if item.strip()]

    return []


def list_to_str(lst):
    if not lst:
        return None
    return ", ".join(lst)


def is_hold_keyword(keyword):
    if pd.isna(keyword):
        return False
    k = str(keyword).strip()
    if k == "":
        return False
    return k.lower() in HOLD_KEYWORDS_LOWER


# =========================
# 4. 원본 엑셀 읽기
# =========================
xls = pd.ExcelFile(file_path)

df_job = pd.read_excel(file_path, sheet_name=sheet_job)
df_norm_freq = pd.read_excel(file_path, sheet_name=sheet_norm_freq)
df_hold_freq = pd.read_excel(file_path, sheet_name=sheet_hold_freq)

# =========================
# 5. job_posting_processed 처리
# =========================
df_job["TECH_STACK_NORMALIZED_LIST"] = df_job["TECH_STACK_NORMALIZED_LIST"].apply(parse_list_cell)

if "TECH_STACK_HOLD_LIST" not in df_job.columns:
    df_job["TECH_STACK_HOLD_LIST"] = [[] for _ in range(len(df_job))]
else:
    df_job["TECH_STACK_HOLD_LIST"] = df_job["TECH_STACK_HOLD_LIST"].apply(parse_list_cell)

new_normalized_lists = []
new_hold_lists = []

for _, row in df_job.iterrows():
    normalized_list = row["TECH_STACK_NORMALIZED_LIST"]
    hold_list = row["TECH_STACK_HOLD_LIST"]

    moved_to_hold = [kw for kw in normalized_list if is_hold_keyword(kw)]
    remained_normalized = [kw for kw in normalized_list if not is_hold_keyword(kw)]

    updated_hold = hold_list + moved_to_hold

    # 중복 제거, 순서 유지
    updated_hold = list(dict.fromkeys(updated_hold))
    remained_normalized = list(dict.fromkeys(remained_normalized))

    new_normalized_lists.append(remained_normalized)
    new_hold_lists.append(updated_hold)

df_job["TECH_STACK_NORMALIZED_LIST"] = new_normalized_lists
df_job["TECH_STACK_HOLD_LIST"] = new_hold_lists
df_job["TECH_STACK_NORMALIZED"] = df_job["TECH_STACK_NORMALIZED_LIST"].apply(list_to_str)
df_job["TECH_STACK_HOLD"] = df_job["TECH_STACK_HOLD_LIST"].apply(list_to_str)
df_job["HAS_HOLD"] = df_job["TECH_STACK_HOLD_LIST"].apply(lambda x: len(x) > 0)

# 저장용 문자열 변환
df_job["TECH_STACK_NORMALIZED_LIST"] = df_job["TECH_STACK_NORMALIZED_LIST"].apply(str)
df_job["TECH_STACK_HOLD_LIST"] = df_job["TECH_STACK_HOLD_LIST"].apply(str)

# =========================
# 6. normalized_freq -> hold_freq 이동
# =========================
# normalized_freq의 tech_stack 기준으로 hold 대상 분리
move_mask = df_norm_freq["tech_stack"].apply(is_hold_keyword)

move_df = df_norm_freq[move_mask].copy()
remain_df = df_norm_freq[~move_mask].copy()

# hold_freq에 합칠 형태 맞추기
# hold_freq 컬럼명: hold_token
# 빈도 컬럼은 normalized_freq와 동일하다고 가정
move_df = move_df.rename(columns={"tech_stack": "hold_token"})

# hold_freq에 없는 컬럼은 맞춰주기
for col in df_hold_freq.columns:
    if col not in move_df.columns:
        move_df[col] = None

# move_df에만 있고 hold_freq에는 없는 컬럼은 제거
move_df = move_df[df_hold_freq.columns]

# 기존 hold_freq + 이동 데이터 합치기
merged_hold_freq = pd.concat([df_hold_freq, move_df], ignore_index=True)

# =========================
# 7. hold_freq 중복 키워드 합산
# =========================
# hold_token 제외 숫자형 컬럼 합산
group_cols = ["hold_token"]

agg_dict = {}
for col in merged_hold_freq.columns:
    if col == "hold_token":
        continue
    if pd.api.types.is_numeric_dtype(merged_hold_freq[col]):
        agg_dict[col] = "sum"
    else:
        agg_dict[col] = "first"

df_hold_freq_final = merged_hold_freq.groupby(group_cols, as_index=False).agg(agg_dict)
df_hold_freq_final = df_hold_freq_final.sort_values(by="count", ascending=False).reset_index(drop=True)

# =========================
# 8. normalized_freq 결과 반영
# =========================
df_norm_freq_final = remain_df.copy()

# =========================
# 9. 엑셀 저장
# =========================
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for s in xls.sheet_names:
        if s == sheet_job:
            df_job.to_excel(writer, sheet_name=s, index=False)
        elif s == sheet_norm_freq:
            df_norm_freq_final.to_excel(writer, sheet_name=s, index=False)
        elif s == sheet_hold_freq:
            df_hold_freq_final.to_excel(writer, sheet_name=s, index=False)
        else:
            temp_df = pd.read_excel(file_path, sheet_name=s)
            temp_df.to_excel(writer, sheet_name=s, index=False)

print("완료")
print(f"저장 파일: {output_path}")
print(f"job_posting_processed에서 HOLD 처리된 행 수: {df_job['HAS_HOLD'].sum()}")
print(f"normalized_freq -> hold_freq 이동 키워드 수: {len(move_df)}")
print(f"normalized_freq 남은 행 수: {len(df_norm_freq_final)}")
print(f"hold_freq 최종 행 수: {len(df_hold_freq_final)}")

완료
저장 파일: ./data/공고원본_POSTING_ID포함_정리결과_hold반영.xlsx
job_posting_processed에서 HOLD 처리된 행 수: 226
normalized_freq -> hold_freq 이동 키워드 수: 12
normalized_freq 남은 행 수: 201
hold_freq 최종 행 수: 37


In [8]:
import pandas as pd

file_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영.xlsx"
output_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx"

sheet_name = "normalized_freq"

# 1. 수동 분류 기준
manual_category_map = {
    # 프론트엔드
    "JavaScript": "프론트엔드",
    "React": "프론트엔드",
    "HTML": "프론트엔드",
    "CSS": "프론트엔드",
    "JSP": "프론트엔드",
    "jQuery": "프론트엔드",
    "Ajax": "프론트엔드",
    "Bootstrap": "프론트엔드",
    "TypeScript": "프론트엔드",
    "Vue.js": "프론트엔드",

    # 백엔드
    "Java": "백엔드",
    "MySQL": "백엔드",
    "Oracle": "백엔드",
    "Spring": "백엔드",
    "SQL": "백엔드",
    "Node.js": "백엔드",
    "Spring Boot": "백엔드",
    "API": "백엔드",
    "C#": "백엔드",
    "C++": "백엔드",
    "DBMS": "백엔드",
    "RDBMS": "백엔드",
    "PHP": "백엔드",
    "C": "백엔드",
    "REST API": "백엔드",
    "MSSQL": "백엔드",

    # 인공지능/데이터
    "Python": "인공지능/데이터",
    "AI": "인공지능/데이터",
    "TensorFlow": "인공지능/데이터",
    "PyTorch": "인공지능/데이터",
    "머신러닝": "인공지능/데이터",
    "딥러닝": "인공지능/데이터",
    "LLM": "인공지능/데이터",
    "OpenCV": "인공지능/데이터",
    "Pandas": "인공지능/데이터",

    # 인프라/보안
    "AWS": "인프라/보안",
    "Linux": "인프라/보안",
    "Docker": "인프라/보안",
    "클라우드": "인프라/보안",

    # 모바일
    "Android": "모바일",
    "Flutter": "모바일",
    "Kotlin": "모바일",
}

# 2. 파일 읽기
xls = pd.ExcelFile(file_path)
df = pd.read_excel(file_path, sheet_name=sheet_name)

# 3. count >= 10만 분류
target_mask = df["count"] >= 10
df_target = df[target_mask].copy()

df_target["category"] = df_target["tech_stack"].map(manual_category_map)

# 혹시 누락된 키워드가 있는지 확인
unmatched = df_target[df_target["category"].isna()].copy()

print("count >= 10 대상 개수:", len(df_target))
print("미분류 개수:", len(unmatched))
if len(unmatched) > 0:
    print(unmatched[["tech_stack", "count"]])

# 4. 원본 df에도 category 붙이기
df["category"] = None
df.loc[target_mask, "category"] = df_target["category"].values

# 5. 저장
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for s in xls.sheet_names:
        if s == sheet_name:
            df.to_excel(writer, sheet_name=s, index=False)
        else:
            temp_df = pd.read_excel(file_path, sheet_name=s)
            temp_df.to_excel(writer, sheet_name=s, index=False)

    # 분류 대상만 따로 보기 쉽게 새 시트 저장
    df_target.to_excel(writer, sheet_name="normalized_freq_labeled", index=False)

print("저장 완료:", output_path)
df_target = df[df["count"] >= 10].copy()
df_target["category"] = df_target["tech_stack"].map(manual_category_map)
print(df_target.sort_values(["category", "count"], ascending=[True, False]))

count >= 10 대상 개수: 42
미분류 개수: 0
저장 완료: ./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx
     tech_stack  count  category
24      Android     33       모바일
28      Flutter     20       모바일
31       Kotlin     19       모바일
0          Java    331       백엔드
6         MySQL    127       백엔드
8        Oracle    109       백엔드
10       Spring    105       백엔드
11          SQL    101       백엔드
13      Node.js     92       백엔드
14  Spring Boot     72       백엔드
15          API     65       백엔드
22           C#     36       백엔드
23          C++     36       백엔드
25         DBMS     24       백엔드
26        RDBMS     23       백엔드
29          PHP     20       백엔드
30            C     19       백엔드
32     REST API     18       백엔드
35        MSSQL     12       백엔드
1        Python    263  인공지능/데이터
17           AI     52  인공지능/데이터
18   TensorFlow     51  인공지능/데이터
19      PyTorch     49  인공지능/데이터
20         머신러닝     40  인공지능/데이터
21          딥러닝     37  인공지능/데이터
33          LLM     17  인공지능/데이터
38       OpenCV     1

In [10]:
import pandas as pd
import re

file_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영.xlsx"
output_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx"

sheet_name = "normalized_freq"

# =========================
# 1. count >= 10 수동 분류 기준
# =========================
manual_category_map = {
    # 프론트엔드
    "JavaScript": "프론트엔드",
    "React": "프론트엔드",
    "HTML": "프론트엔드",
    "CSS": "프론트엔드",
    "JSP": "프론트엔드",
    "jQuery": "프론트엔드",
    "Ajax": "프론트엔드",
    "Bootstrap": "프론트엔드",
    "TypeScript": "프론트엔드",
    "Vue.js": "프론트엔드",

    # 백엔드
    "Java": "백엔드",
    "MySQL": "백엔드",
    "Oracle": "백엔드",
    "Spring": "백엔드",
    "SQL": "백엔드",
    "Node.js": "백엔드",
    "Spring Boot": "백엔드",
    "API": "백엔드",
    "C#": "백엔드",
    "C++": "백엔드",
    "DBMS": "백엔드",
    "RDBMS": "백엔드",
    "PHP": "백엔드",
    "C": "백엔드",
    "REST API": "백엔드",
    "MSSQL": "백엔드",

    # 인공지능/데이터
    "Python": "인공지능/데이터",
    "AI": "인공지능/데이터",
    "TensorFlow": "인공지능/데이터",
    "PyTorch": "인공지능/데이터",
    "머신러닝": "인공지능/데이터",
    "딥러닝": "인공지능/데이터",
    "LLM": "인공지능/데이터",
    "OpenCV": "인공지능/데이터",
    "Pandas": "인공지능/데이터",

    # 인프라/보안
    "AWS": "인프라/보안",
    "Linux": "인프라/보안",
    "Docker": "인프라/보안",
    "클라우드": "인프라/보안",

    # 모바일
    "Android": "모바일",
    "Flutter": "모바일",
    "Kotlin": "모바일",
}

# =========================
# 2. 규칙 기반 자동 분류 사전
#    count < 10 대상에 적용
# =========================
category_rules = {
    "프론트엔드": [
        "javascript", "typescript", "react", "vue", "angular", "svelte",
        "html", "css", "scss", "sass", "less",
        "jsp", "thymeleaf", "jquery", "ajax", "bootstrap", "tailwind",
        "frontend", "front-end", "web ui", "webui", "next.js", "nuxt"
    ],
    "백엔드": [
        "java", "spring", "spring boot", "node", "node.js", "express",
        "python",  # 단, 아래 예외 처리에서 AI/데이터 쪽 키워드와 같이 보이면 AI/데이터로 보정 가능
        "php", "c#", "c++", "c language", "golang", "go", "kotlin server",
        "sql", "mysql", "oracle", "postgres", "postgresql", "mariadb", "mongodb",
        "redis", "db", "dbms", "rdbms",
        "api", "rest", "rest api", "graphql", "backend", "back-end",
        "servlet", "tomcat", "jpa", "hibernate", "mybatis"
    ],
    "인공지능/데이터": [
        "ai", "ml", "dl", "llm", "nlp", "cv",
        "machine learning", "deep learning",
        "머신러닝", "딥러닝", "자연어처리", "컴퓨터비전",
        "tensorflow", "pytorch", "keras", "scikit", "sklearn",
        "opencv", "pandas", "numpy", "matplotlib", "seaborn",
        "data", "dataset", "datamining", "data mining",
        "hadoop", "spark", "hive", "분석", "통계", "예측", "모델링"
    ],
    "인프라/보안": [
        "aws", "azure", "gcp", "cloud", "클라우드",
        "docker", "kubernetes", "k8s", "jenkins", "terraform",
        "linux", "unix", "server", "nginx", "apache",
        "devops", "infra", "인프라",
        "security", "보안", "network", "네트워크",
        "iam", "vpn", "firewall", "ci/cd", "cicd"
    ],
    "모바일": [
        "android", "ios", "swift", "objective-c", "objective c",
        "kotlin", "flutter", "react native", "xamarin",
        "mobile", "app", "앱", "모바일"
    ]
}

# =========================
# 3. 자동 분류 함수
# =========================
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text

def auto_classify_keyword(keyword):
    """
    규칙 기반 자동 분류
    - 규칙 매칭되면 해당 카테고리 반환
    - 아무 규칙에도 안 걸리면 '기타'
    """
    kw = normalize_text(keyword)

    if kw == "":
        return "기타"

    # -------------------------
    # 예외 우선 처리
    # -------------------------
    # Python은 단독이면 AI/데이터로 보는 쪽이 현재 프로젝트 맥락상 자연스러움
    if kw == "python":
        return "인공지능/데이터"

    # C / C++ / C# 계열은 일단 백엔드로
    if kw in ["c", "c++", "c#"]:
        return "백엔드"

    # SQL 계열은 백엔드
    if kw in ["sql", "mysql", "oracle", "mssql", "postgresql", "mariadb"]:
        return "백엔드"

    # -------------------------
    # 규칙 점수 계산
    # -------------------------
    score = {
        "프론트엔드": 0,
        "백엔드": 0,
        "인공지능/데이터": 0,
        "인프라/보안": 0,
        "모바일": 0
    }

    for category, patterns in category_rules.items():
        for pattern in patterns:
            if pattern in kw:
                score[category] += 1

    # 매칭이 전혀 없으면 기타
    max_score = max(score.values())
    if max_score == 0:
        return "기타"

    # 동점이면 우선순위 부여
    priority = ["인공지능/데이터", "모바일", "인프라/보안", "프론트엔드", "백엔드"]
    candidates = [cat for cat, s in score.items() if s == max_score]

    for p in priority:
        if p in candidates:
            return p

    return "기타"

# =========================
# 4. 파일 읽기
# =========================
xls = pd.ExcelFile(file_path)
df = pd.read_excel(file_path, sheet_name=sheet_name)

# =========================
# 5. 최종 분류 함수
# =========================
def classify_row(row):
    tech = row["tech_stack"]
    cnt = row["count"]

    # count >= 10 은 수동 분류 기준 우선
    if cnt >= 10:
        return manual_category_map.get(tech, auto_classify_keyword(tech))

    # count < 10 은 규칙 기반 자동 분류
    return auto_classify_keyword(tech)

df["category"] = df.apply(classify_row, axis=1)

# =========================
# 6. 분류 방식 표시 컬럼(선택)
# =========================
def classify_source(row):
    tech = row["tech_stack"]
    cnt = row["count"]

    if cnt >= 10 and tech in manual_category_map:
        return "수동분류"
    elif row["category"] == "기타":
        return "기타처리"
    else:
        return "자동분류"

df["category_source"] = df.apply(classify_source, axis=1)

# =========================
# 7. 결과 확인용 출력
# =========================
print("전체 행 수:", len(df))
print("\n카테고리별 개수")
print(df["category"].value_counts(dropna=False))

print("\n기타로 분류된 키워드")
print(df[df["category"] == "기타"][["tech_stack", "count"]].sort_values("count", ascending=False))

# =========================
# 8. 저장
# =========================
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for s in xls.sheet_names:
        if s == sheet_name:
            df.to_excel(writer, sheet_name=s, index=False)
        else:
            temp_df = pd.read_excel(file_path, sheet_name=s)
            temp_df.to_excel(writer, sheet_name=s, index=False)

    # 확인용 시트
    df[df["count"] >= 10].sort_values(["category", "count"], ascending=[True, False]) \
        .to_excel(writer, sheet_name="count10_over_labeled", index=False)

    df[df["count"] < 10].sort_values(["category", "count"], ascending=[True, False]) \
        .to_excel(writer, sheet_name="count10_under_auto", index=False)

    df[df["category"] == "기타"].sort_values("count", ascending=False) \
        .to_excel(writer, sheet_name="etc_keywords", index=False)

print("저장 완료:", output_path)

전체 행 수: 201

카테고리별 개수
category
기타          108
백엔드          34
인공지능/데이터     22
프론트엔드        15
인프라/보안       15
모바일           7
Name: count, dtype: int64

기타로 분류된 키워드
      tech_stack  count
42           IoT      9
50             R      8
51   전자정부표준프레임워크      8
48          Jira      8
53       ChatGPT      6
..           ...    ...
159      Laravel      1
160        LiDAR      1
161        Llama      1
162          MCP      1
200   eXBuilder6      1

[108 rows x 2 columns]
저장 완료: ./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx


In [11]:
import oracledb

oracledb.init_oracle_client(
    lib_dir=r"C:\instantclient-basic-windows.x64-23.26.1.0.0\instantclient_23_0"
)

In [12]:
import pandas as pd
import oracledb

# =========================
# 1. 엑셀 파일 불러오기
# =========================
excel_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx"
sheet_name = "job_posting_processed"

df = pd.read_excel(excel_path, sheet_name=sheet_name)

# 필요한 컬럼만 추출
update_df = df[["POSTING_ID", "TECH_STACK_NORMALIZED"]].copy()

# POSTING_ID가 없는 행 제거
update_df = update_df.dropna(subset=["POSTING_ID"])

# NaN -> None 처리 (DB에 NULL로 들어가게)
update_df["TECH_STACK_NORMALIZED"] = update_df["TECH_STACK_NORMALIZED"].where(
    pd.notna(update_df["TECH_STACK_NORMALIZED"]), None
)

# POSTING_ID 정수형 변환
update_df["POSTING_ID"] = update_df["POSTING_ID"].astype(int)

print(f"업데이트 대상 행 수: {len(update_df)}")


# =========================
# 2. 오라클 DB 연결
# =========================
conn = oracledb.connect(
    user='campus_24KDT_LI8_p2_1',
    password='smhrd1',
    dsn="project-db-campus.smhrd.com:1524/xe"
)

cur = conn.cursor()


# =========================
# 3. UPDATE 실행
# =========================
sql = """
UPDATE JOB_POSTING
SET TECH_STACK = :tech_stack
WHERE POSTING_ID = :posting_id
"""

data = [
    {
        "tech_stack": row["TECH_STACK_NORMALIZED"],
        "posting_id": row["POSTING_ID"]
    }
    for _, row in update_df.iterrows()
]

cur.executemany(sql, data)
conn.commit()

print(f"업데이트 완료: {cur.rowcount}건")


# =========================
# 4. 연결 종료
# =========================
cur.close()
conn.close()

업데이트 대상 행 수: 884
업데이트 완료: 884건


In [13]:
import pandas as pd
import oracledb

# =========================
# 1. 엑셀 파일 읽기
# =========================
excel_path = "./data/공고원본_POSTING_ID포함_정리결과_hold반영_카테고리분류.xlsx"
sheet_name = "normalized_freq"

df = pd.read_excel(excel_path, sheet_name=sheet_name)

# 필요한 컬럼만 추출
stack_df = df[["tech_stack", "category"]].copy()

# tech_stack이 비어 있는 행 제거
stack_df = stack_df.dropna(subset=["tech_stack"])

# 문자열 정리
stack_df["tech_stack"] = stack_df["tech_stack"].astype(str).str.strip()
stack_df["category"] = stack_df["category"].astype(str).str.strip()

# =========================
# 2. 카테고리명 -> CATEGORY_ID 매핑
# =========================
category_map = {
    "프론트엔드": 1,
    "백엔드": 2,
    "인공지능/데이터": 3,
    "인프라/보안": 4,
    "모바일": 5,
    "기타": 6
}

stack_df["category_id"] = stack_df["category"].map(category_map)

# 매핑 안 된 값 확인
unmapped = stack_df[stack_df["category_id"].isna()]
if len(unmapped) > 0:
    print("카테고리 매핑 실패 데이터:")
    print(unmapped[["tech_stack", "category"]].drop_duplicates())
    raise ValueError("category_id로 변환되지 않은 category 값이 있음")

# =========================
# 3. STACK_ID 부여
# =========================
stack_df = stack_df.reset_index(drop=True)
stack_df["stack_id"] = stack_df.index + 1

# DB에 넣을 컬럼 순서 정리
insert_df = stack_df[["stack_id", "tech_stack", "category_id"]].copy()

print("삽입 대상 건수:", len(insert_df))
print(insert_df.head())

# =========================
# 4. 오라클 DB 연결
# =========================
conn = oracledb.connect(
    user='campus_24KDT_LI8_p2_1',
    password='smhrd1',
    dsn="project-db-campus.smhrd.com:1524/xe"
)

cur = conn.cursor()

# =========================
# 5. INSERT 실행
# =========================
sql = """
INSERT INTO TECH_STACK (STACK_ID, STACK_NAME, CATEGORY_ID)
VALUES (:1, :2, :3)
"""

data = list(insert_df.itertuples(index=False, name=None))

cur.executemany(sql, data)
conn.commit()

print("삽입 완료:", cur.rowcount, "건")

# =========================
# 6. 연결 종료
# =========================
cur.close()
conn.close()

삽입 대상 건수: 201
   stack_id  tech_stack  category_id
0         1        Java            2
1         2      Python            3
2         3  JavaScript            1
3         4       React            1
4         5        HTML            1
삽입 완료: 201 건
